In [0]:
%run ../_shared/00_project_config

In [0]:
from pyspark.sql import functions as F

In [0]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

SOURCE_SYSTEM = "football-data.org"

RAW_TEAMS_ROOT = (
    f"{FOOTBALL_DATA_ORG_RAW_ROOT}/teams"
)

BRONZE_TABLE = (
    f"{BRONZE_SCHEMA}."
    "football_data_org_teams"
)


RAW_TEAM_FILE_NAMES = {
    "PL": "pl_premier_league.json",
    "PD": "pd_la_liga.json",
    "BL1": "bl1_bundesliga.json",
    "SA": "sa_serie_a.json",
    "FL1": "fl1_ligue_1.json",
}

# -----------------------------------------------------------------------------
# Load football-data.org team JSON files into Bronze
# -----------------------------------------------------------------------------
#
# Each raw JSON file contains a response envelope with fields such as:
#
#   count
#   filters
#   competition
#   season
#   teams[]
#
# Bronze removes only the API response envelope by exploding teams[].
#
# Resulting Bronze grain:
#
#   one row = one football-data.org team within the competition snapshot
#
# Source field names and nested source structures are preserved.
#
# Technical metadata:
#
#   _league
#   _ingested_at
#   _source_file
#   _source_system
# -----------------------------------------------------------------------------

loaded_file_count = 0
loaded_team_count = 0


for competition_code, league in FOOTBALL_DATA_ORG_COMPETITIONS.items():

    file_name = RAW_TEAM_FILE_NAMES[
        competition_code
    ]

    file_path = (
        f"{RAW_TEAMS_ROOT}/"
        f"{file_name}"
    )

    # -------------------------------------------------------------------------
    # Read raw JSON
    # -------------------------------------------------------------------------
    # Raw files are pretty-printed JSON documents, therefore multiLine
    # must be enabled.
    # -------------------------------------------------------------------------

    source_df = (
        spark.read
        .option("multiLine", "true")
        .json(file_path)
    )

    # -------------------------------------------------------------------------
    # Explode teams[]
    # -------------------------------------------------------------------------
    #
    # Keep competition and season context from the response envelope.
    # -------------------------------------------------------------------------

    bronze_df = (
        source_df
        .select(
            "competition",
            "season",
            F.explode("teams").alias("team"),
            F.col("_metadata.file_path").alias("_source_file"),
        )
        .select(
            "competition",
            "season",
            "team.*",
            "_source_file",
        )
        .withColumn(
            "_league",
            F.lit(league),
        )
        .withColumn(
            "_ingested_at",
            F.current_timestamp(),
        )
        .withColumn(
            "_source_system",
            F.lit(SOURCE_SYSTEM),
        )
    )

    team_count = bronze_df.count()

    # -------------------------------------------------------------------------
    # Write to Bronze Delta table
    # -------------------------------------------------------------------------

    (
        bronze_df.write
        .format("delta")
        .mode("append")
        .option(
            "mergeSchema",
            "true",
        )
        .saveAsTable(
            BRONZE_TABLE
        )
    )

    loaded_file_count += 1
    loaded_team_count += team_count

    print(
        f"Loaded football-data.org Bronze teams file "
        f"{loaded_file_count}: "
        f"{competition_code} / {league} | "
        f"teams={team_count} "
        f"-> {BRONZE_TABLE}"
    )

# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {loaded_file_count} "
    "football-data.org team source files into Bronze. "
    f"Total teams loaded: {loaded_team_count}."
)